# 10 · Testing, Security, and Reproducibility

Run the automated quality gates, inspect ignored artifacts, and perform an explicit source secret scan.

**How to use this notebook**

1. Run cells from top to bottom.
2. Start with the CI profile; switch to `portfolio` only after the workflow is clear.
3. Change one parameter at a time and record the effect rather than chasing a high score.

> All records are deterministic synthetic data. Results are educational evidence, not a
> production or financial decision system.

In [ ]:
# Shared local/Colab bootstrap. Run this cell first.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "codex/colab-notebook-first",
                "--single-branch",
                "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
    %pip install -q -e ".[full,dev]"
else:
    here = Path.cwd().resolve()
    if here.name == "notebooks":
        os.chdir(here.parent)

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").exists(), "Please open the repository root or notebooks folder."
print(f"environment={'Colab' if IN_COLAB else 'local/VS Code'}")
print(f"repository={ROOT}")

In [ ]:
!python -m pytest -q
!python -m ruff check src scripts tests

In [ ]:
import re
from pathlib import Path

patterns = [
    re.compile(r"sk-[A-Za-z0-9_-]{20,}"),
    re.compile(r"AKIA[0-9A-Z]{16}"),
    re.compile(r"BEGIN (?:RSA|OPENSSH|EC) PRIVATE KEY"),
]
findings = []
for path in Path(".").rglob("*"):
    relative = str(path)
    if not path.is_file() or any(
        part in relative for part in (".git", ".venv", "data/generated", "artifacts")
    ):
        continue
    if path.suffix.lower() in {".py", ".md", ".toml", ".yaml", ".yml", ".json", ".sql"}:
        text = path.read_text(errors="ignore")
        if any(pattern.search(text) for pattern in patterns):
            findings.append(relative)
assert not findings, findings
print("Secret-pattern scan clean")

In [ ]:
print(Path(".gitignore").read_text())
print(Path("reports/security.md").read_text())